In [1]:
from google.colab import files
uploaded = files.upload()


Saving cleaned_customer_churn.csv to cleaned_customer_churn (1).csv


In [3]:
import pandas as pd
df=pd.read_csv('/content/cleaned_customer_churn (1).csv')
print("Dataset loaded successfully!")
print ("Dataset shape:", df. shape)
display(df. head())
print("Column names:",df. columns.tolist())

Dataset loaded successfully!
Dataset shape: (15, 11)


,CustomerID,Gender,Age,TenureMonths,SubscriptionType,MonthlyCharges,TotalCharges,ContractType,SupportTickets,PaymentMethod,Churn
0,CUST-1001,Female,34,12,Basic,49.99,599.88,Month-to-Month,3,Credit Card,Yes
1,CUST-1002,Male,45,24,Pro,79.99,1919.76,One Year,1,Bank Transfer,No
2,CUST-1003,Female,29,6,Basic,49.99,299.94,Month-to-Month,5,UPI,Yes
3,CUST-1004,Male,52,36,Enterprise,149.99,5399.64,Two Year,0,Credit Card,No
4,CUST-1005,Female,41,8,Pro,79.99,639.92,Month-to-Month,4,Debit Card,Yes


Column names: ['CustomerID', 'Gender', 'Age', 'TenureMonths', 'SubscriptionType', 'MonthlyCharges', 'TotalCharges', 'ContractType', 'SupportTickets', 'PaymentMethod', 'Churn']


In [4]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove duplicate customer records
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Display missing values
print("\nMissing values:")
print(df.isnull().sum())

# Display data types
print("\nData types:")
print(df.dtypes)

# Display numerical summary
display(df.describe(include="all").T)

Duplicate rows: 0

Missing values:
CustomerID          0
Gender              0
Age                 0
TenureMonths        0
SubscriptionType    0
MonthlyCharges      0
TotalCharges        0
ContractType        0
SupportTickets      0
PaymentMethod       0
Churn               0
dtype: int64

Data types:
CustomerID           object
Gender               object
Age                   int64
TenureMonths          int64
SubscriptionType     object
MonthlyCharges      float64
TotalCharges        float64
ContractType         object
SupportTickets        int64
PaymentMethod        object
Churn                object
dtype: object


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
CustomerID,15,15,CUST-1001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,15,2,Female,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,15.0,NaN,NaN,NaN,39.066667,11.392144,23.0,30.0,38.0,47.0,60.0
TenureMonths,15.0,NaN,NaN,NaN,18.8,14.333776,2.0,7.0,15.0,27.0,48.0
SubscriptionType,15,3,Basic,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MonthlyCharges,15.0,NaN,NaN,NaN,84.656667,42.739521,49.99,49.99,79.99,114.99,149.99
TotalCharges,15.0,NaN,NaN,NaN,2088.478667,2407.365767,99.98,449.91,1099.78,3209.73,7199.52
ContractType,15,3,Month-to-Month,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SupportTickets,15.0,NaN,NaN,NaN,2.533333,1.9223,0.0,1.0,2.0,4.0,6.0
PaymentMethod,15,4,Credit Card,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Fields required for this analysis
required = [
    "CustomerID",
    "TenureMonths",
    "MonthlyCharges",
    "SupportTickets"
]

missing_columns = [c for c in required if c not in df.columns]

if missing_columns:
    raise ValueError(
        f"Please check your CSV column names: {missing_columns}"
    )

# Convert numerical fields safely
numeric_cols = [
    "TenureMonths",
    "MonthlyCharges",
    "SupportTickets"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Customers:", df["CustomerID"].nunique())
print("Missing scoring fields:")
print(df[numeric_cols].isnull().sum())

Customers: 15
Missing scoring fields:
TenureMonths      0
MonthlyCharges    0
SupportTickets    0
dtype: int64


In [7]:
import pandas as pd
import numpy as np
# Establish a reference value from the supplied dataset
charge_median = df["MonthlyCharges"].median()

# Flag records that need review rather than silently scoring them
df["NeedsDataReview"] = df[
    ["TenureMonths", "MonthlyCharges", "SupportTickets"]
].isnull().any(axis=1)

# Use neutral values for missing fields only to permit calculation.
# The review flag prevents these records being treated as reliable.
tenure = df["TenureMonths"].fillna(0)
tickets = df["SupportTickets"].fillna(0)
charges = df["MonthlyCharges"].fillna(charge_median)

df["RiskScore"] = (
    (tenure.le(12).astype(int) * 35)
    + (tickets.ge(3).astype(int) * 35)
    + (tickets.between(1, 2).astype(int) * 15)
    + (charges.ge(charge_median).astype(int) * 20)
)

# Do not assign a final risk category to records needing review
df["RiskLevel"] = np.select(
    [
        df["NeedsDataReview"],
        df["RiskScore"].ge(60),
        df["RiskScore"].ge(30)
    ],
    [
        "Needs Review",
        "High",
        "Medium"
    ],
    default="Low"
)

print("Monthly charge median:", round(charge_median, 2))

display(
    df[
        ["CustomerID", "TenureMonths", "MonthlyCharges",
         "SupportTickets", "RiskScore", "RiskLevel",
         "NeedsDataReview"]
    ].head(10)
)

Monthly charge median: 79.99


,CustomerID,TenureMonths,MonthlyCharges,SupportTickets,RiskScore,RiskLevel,NeedsDataReview
0,CUST-1001,12,49.99,3,70,High,False
1,CUST-1002,24,79.99,1,35,Medium,False
2,CUST-1003,6,49.99,5,70,High,False
3,CUST-1004,36,149.99,0,20,Low,False
4,CUST-1005,8,79.99,4,90,High,False
5,CUST-1006,18,49.99,2,15,Low,False
6,CUST-1007,3,49.99,6,70,High,False
7,CUST-1008,48,149.99,1,35,Medium,False
8,CUST-1009,15,79.99,2,35,Medium,False
9,CUST-1010,4,49.99,4,70,High,False


In [14]:
import pandas as pd

# Step 1: Create customer segments
def assign_segment(row):
    if row["NeedsDataReview"]:
        return "Data Review Required"

    if row["RiskLevel"] == "High" and row["TenureMonths"] <= 12:
        return "New Customers Needing Support"

    if row["RiskLevel"] == "High":
        return "High-Risk Customers"

    if row["TenureMonths"] >= 25:
        return "Long-Tenure Customers"

    return "Regular Monitoring"


# Step 2: Apply segmentation to the main dataset
df["CustomerSegment"] = df.apply(assign_segment, axis=1)

# Step 3: Display the segmented customer table
display(
    df[
        [
            "CustomerID",
            "TenureMonths",
            "SupportTickets",
            "MonthlyCharges",
            "RiskScore",
            "RiskLevel",
            "CustomerSegment"
        ]
    ].sort_values("RiskScore", ascending=False)
)
print("Customer Segment Summary")
display(
    df.groupby("CustomerSegment")
    .agg(
        Customers=("CustomerID", "nunique"),
        AverageRiskScore=("RiskScore","mean"),
        AverageTenure=("TenureMonths","mean")
    )
.round(2)
)

,CustomerID,TenureMonths,SupportTickets,MonthlyCharges,RiskScore,RiskLevel,CustomerSegment
4,CUST-1005,8,4,79.99,90,High,New Customers Needing Support
2,CUST-1003,6,5,49.99,70,High,New Customers Needing Support
0,CUST-1001,12,3,49.99,70,High,New Customers Needing Support
6,CUST-1007,3,6,49.99,70,High,New Customers Needing Support
9,CUST-1010,4,4,49.99,70,High,New Customers Needing Support
13,CUST-1014,2,5,49.99,70,High,New Customers Needing Support
11,CUST-1012,14,3,79.99,55,Medium,Regular Monitoring
14,CUST-1015,40,1,149.99,35,Medium,Long-Tenure Customers
8,CUST-1009,15,2,79.99,35,Medium,Regular Monitoring
1,CUST-1002,24,1,79.99,35,Medium,Regular Monitoring


Customer Segment Summary


,Customers,AverageRiskScore,AverageTenure
CustomerSegment,,,
Long-Tenure Customers,4,27.50,38.50
New Customers Needing Support,6,73.33,5.83
Regular Monitoring,5,31.00,18.60


In [15]:
if "Churn" in df.columns:

    churn_clean = (
        df["Churn"].astype("string").str.strip().str.lower()
    )

    df["ChurnFlag"] = churn_clean.map({
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0
    })

    print("Unrecognized or missing churn labels:",
          df["ChurnFlag"].isna().sum())

    validation_data = df.dropna(
        subset=["ChurnFlag"]
    ).copy()

    validation_data = validation_data[
        ~validation_data["NeedsDataReview"]
    ]

    if len(validation_data) > 0:
        validation_data["ChurnFlag"] = (
            validation_data["ChurnFlag"].astype(int)
        )

        print("\nHistorical churn rate by risk level:")
        display(
            validation_data.groupby("RiskLevel")
            .agg(
                Customers=("CustomerID", "nunique"),
                HistoricalChurnRate=("ChurnFlag", "mean"),
                AverageRiskScore=("RiskScore", "mean")
            )
            .assign(
                HistoricalChurnRate=lambda x:
                    (x["HistoricalChurnRate"] * 100).round(2)
                .astype(str) + "%"
            )
        )

        print("\nHistorical churn rate by segment:")
        display(
            validation_data.groupby("CustomerSegment")
            .agg(
                Customers=("CustomerID", "nunique"),
                HistoricalChurnRate=("ChurnFlag", "mean")
            )
        )
else:
    print(
        "No Churn column found. Risk scoring was created, "
        "but historical churn validation cannot be performed."
    )

Unrecognized or missing churn labels: 0

Historical churn rate by risk level:


,Customers,HistoricalChurnRate,AverageRiskScore
RiskLevel,,,
High,6,100.0%,73.333333
Low,4,0.0%,17.500000
Medium,5,20.0%,39.000000



Historical churn rate by segment:


,Customers,HistoricalChurnRate
CustomerSegment,,
Long-Tenure Customers,4,0.0
New Customers Needing Support,6,1.0
Regular Monitoring,5,0.2


In [16]:
segment_summary = (
    df.groupby("CustomerSegment", dropna=False)
    .agg(
        CustomerCount=("CustomerID", "nunique"),
        AverageTenure=("TenureMonths", "mean"),
        AverageMonthlyCharges=("MonthlyCharges", "mean"),
        AverageSupportTickets=("SupportTickets", "mean"),
        AverageRiskScore=("RiskScore", "mean")
    )
    .round(2)
    .reset_index()
)

display(segment_summary)

,CustomerSegment,CustomerCount,AverageTenure,AverageMonthlyCharges,AverageSupportTickets,AverageRiskScore
0,Long-Tenure Customers,4,38.50,149.99,0.5,27.50
1,New Customers Needing Support,6,5.83,54.99,4.5,73.33
2,Regular Monitoring,5,18.60,67.99,1.8,31.00


In [18]:
recommendations = {
    "New Customers Needing Support":
        "Provide onboarding assistance, check early issues, "
        "and follow up on unresolved tickets.",

    "High-Risk Customers":
        "Review repeated support issues, contact customers, "
        "and offer a suitable plan or service resolution.",

    "Long-Tenure Customers":
        "Recognize loyalty, gather feedback, and offer "
        "relevant renewal or loyalty benefits.",

    "Regular Monitoring":
        "Monitor satisfaction, billing concerns, and "
        "changes in support activity.",

    "Data Review Required":
        "Verify missing or invalid customer information "
        "before making a retention decision."
}

df["RecommendedAction"] = df["CustomerSegment"].map(
    recommendations
)

segment_table = df[
    [
        "CustomerID",
        "TenureMonths",
        "MonthlyCharges",
        "SupportTickets",
        "RiskScore",
        "RiskLevel",
        "CustomerSegment",
        "RecommendedAction",
        "NeedsDataReview"
    ]
].copy()

segment_table.to_csv(
    "Customer_Segment_Table.csv", index=False
)

segment_summary.to_csv(
    "Customer_Segment_Summary.csv", index=False
)

print("Exported customer-level segment table.")
print("Exported segment summary.")

Exported customer-level segment table.
Exported segment summary.


In [19]:
print("Missing customer IDs:",
      df["CustomerID"].isna().sum())

print("Duplicate customer IDs:",
      df["CustomerID"].duplicated().sum())

print("Negative tenure values:",
      (df["TenureMonths"] < 0).sum())

print("Negative monthly charges:",
      (df["MonthlyCharges"] < 0).sum())

print("Negative support tickets:",
      (df["SupportTickets"] < 0).sum())

print("Customers needing data review:",
      df["NeedsDataReview"].sum())

print("\nRisk-level counts:")
print(df["RiskLevel"].value_counts(dropna=False))

print("\nSegment counts:")
print(df["CustomerSegment"].value_counts(dropna=False))

Missing customer IDs: 0
Duplicate customer IDs: 0
Negative tenure values: 0
Negative monthly charges: 0
Negative support tickets: 0
Customers needing data review: 0

Risk-level counts:
RiskLevel
High      6
Medium    5
Low       4
Name: count, dtype: int64

Segment counts:
CustomerSegment
New Customers Needing Support    6
Regular Monitoring               5
Long-Tenure Customers            4
Name: count, dtype: int64


In [20]:
print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

print("\nCustomer segments:")
display(df["CustomerSegment"].value_counts().reset_index())

print("\nCustomer records:")
display(df.head())

Dataset shape: (15, 17)

Available columns:
['CustomerID', 'Gender', 'Age', 'TenureMonths', 'SubscriptionType', 'MonthlyCharges', 'TotalCharges', 'ContractType', 'SupportTickets', 'PaymentMethod', 'Churn', 'NeedsDataReview', 'RiskScore', 'RiskLevel', 'ChurnFlag', 'CustomerSegment', 'RecommendedAction']

Customer segments:


,CustomerSegment,count
0,New Customers Needing Support,6
1,Regular Monitoring,5
2,Long-Tenure Customers,4



Customer records:


,CustomerID,Gender,Age,TenureMonths,SubscriptionType,MonthlyCharges,TotalCharges,ContractType,SupportTickets,PaymentMethod,Churn,NeedsDataReview,RiskScore,RiskLevel,ChurnFlag,CustomerSegment,RecommendedAction
0,CUST-1001,Female,34,12,Basic,49.99,599.88,Month-to-Month,3,Credit Card,Yes,False,70,High,1,New Customers Needing Support,"Provide onboarding assistance, check early iss..."
1,CUST-1002,Male,45,24,Pro,79.99,1919.76,One Year,1,Bank Transfer,No,False,35,Medium,0,Regular Monitoring,"Monitor satisfaction, billing concerns, and ch..."
2,CUST-1003,Female,29,6,Basic,49.99,299.94,Month-to-Month,5,UPI,Yes,False,70,High,1,New Customers Needing Support,"Provide onboarding assistance, check early iss..."
3,CUST-1004,Male,52,36,Enterprise,149.99,5399.64,Two Year,0,Credit Card,No,False,20,Low,0,Long-Tenure Customers,"Recognize loyalty, gather feedback, and offer ..."
4,CUST-1005,Female,41,8,Pro,79.99,639.92,Month-to-Month,4,Debit Card,Yes,False,90,High,1,New Customers Needing Support,"Provide onboarding assistance, check early iss..."


In [21]:
validation_data = df.copy()

print("Validation data created successfully.")
print("Shape:", validation_data.shape)


Validation data created successfully.
Shape: (15, 17)


In [22]:
import pandas as pd

segment_table = df[
    [
        "CustomerID",
        "TenureMonths",
        "MonthlyCharges",
        "SupportTickets",
        "RiskScore",
        "RiskLevel",
        "NeedsDataReview",
        "CustomerSegment"
    ]
].copy()

display(
    segment_table.sort_values(
        by="RiskScore",
        ascending=False
    )
)

,CustomerID,TenureMonths,MonthlyCharges,SupportTickets,RiskScore,RiskLevel,NeedsDataReview,CustomerSegment
4,CUST-1005,8,79.99,4,90,High,False,New Customers Needing Support
2,CUST-1003,6,49.99,5,70,High,False,New Customers Needing Support
0,CUST-1001,12,49.99,3,70,High,False,New Customers Needing Support
6,CUST-1007,3,49.99,6,70,High,False,New Customers Needing Support
9,CUST-1010,4,49.99,4,70,High,False,New Customers Needing Support
13,CUST-1014,2,49.99,5,70,High,False,New Customers Needing Support
11,CUST-1012,14,79.99,3,55,Medium,False,Regular Monitoring
14,CUST-1015,40,149.99,1,35,Medium,False,Long-Tenure Customers
8,CUST-1009,15,79.99,2,35,Medium,False,Regular Monitoring
1,CUST-1002,24,79.99,1,35,Medium,False,Regular Monitoring


In [23]:
segment_table.to_csv(
    "/content/customer_segment_table.csv",
    index=False
)

print("Customer segment table saved successfully.")

Customer segment table saved successfully.


In [24]:
from google.colab import files

files.download("/content/customer_segment_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
segment_summary = (
    df.groupby("CustomerSegment", dropna=False)
    .agg(
        CustomerCount=("CustomerID", "nunique"),
        AverageRiskScore=("RiskScore", "mean"),
        AverageTenure=("TenureMonths", "mean"),
        AverageMonthlyCharges=("MonthlyCharges", "mean"),
        AverageSupportTickets=("SupportTickets", "mean")
    )
    .reset_index()
)

segment_summary["AverageRiskScore"] = (
    segment_summary["AverageRiskScore"].round(2)
)

segment_summary["AverageTenure"] = (
    segment_summary["AverageTenure"].round(2)
)

segment_summary["AverageMonthlyCharges"] = (
    segment_summary["AverageMonthlyCharges"].round(2)
)

segment_summary["AverageSupportTickets"] = (
    segment_summary["AverageSupportTickets"].round(2)
)

display(segment_summary)

,CustomerSegment,CustomerCount,AverageRiskScore,AverageTenure,AverageMonthlyCharges,AverageSupportTickets
0,Long-Tenure Customers,4,27.50,38.50,149.99,0.5
1,New Customers Needing Support,6,73.33,5.83,54.99,4.5
2,Regular Monitoring,5,31.00,18.60,67.99,1.8


In [26]:
segment_summary.to_csv(
    "/content/customer_segment_summary.csv",
    index=False
)

print("Segment summary saved successfully.")

Segment summary saved successfully.


In [27]:
print(df.columns.tolist())

['CustomerID', 'Gender', 'Age', 'TenureMonths', 'SubscriptionType', 'MonthlyCharges', 'TotalCharges', 'ContractType', 'SupportTickets', 'PaymentMethod', 'Churn', 'NeedsDataReview', 'RiskScore', 'RiskLevel', 'ChurnFlag', 'CustomerSegment', 'RecommendedAction']


In [28]:
if "ChurnFlag" in df.columns:
    churn_summary = (
        df.groupby("RiskLevel")
        .agg(
            Customers=("CustomerID", "nunique"),
            HistoricalChurnRate=("ChurnFlag", "mean"),
            AverageRiskScore=("RiskScore", "mean")
        )
        .reset_index()
    )

    churn_summary["HistoricalChurnRate"] = (
        churn_summary["HistoricalChurnRate"] * 100
    ).round(2)

    display(churn_summary)

else:
    print(
        "No ChurnFlag column found. "
        "Historical churn validation cannot be performed."
    )

,RiskLevel,Customers,HistoricalChurnRate,AverageRiskScore
0,High,6,100.0,73.333333
1,Low,4,0.0,17.500000
2,Medium,5,20.0,39.000000
